[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/03_linear.ipynb)

# 🟡 Medium: Simple Linear Layer

*Core Ops & Layers*
Implement a **fully-connected layer** as a `flax.nnx.Module`.

$$y = xW + b, \qquad W \in \mathbb{R}^{d_{in} \times d_{out}},\; b \in \mathbb{R}^{d_{out}}$$

### Rules
- Subclass `nnx.Module`; do **not** use `nnx.Linear`
- Signature: `SimpleLinear(din, dout, *, use_bias=True, rngs)`
- Weights stored as `self.w`, bias as `self.b` (or `None` when `use_bias=False`)
- Both must be wrapped in `nnx.Param`
- Initialise `w` with `jax.random.normal(...) / sqrt(din)`; `b` with zeros
- `__call__(x)` maps `(..., din) -> (..., dout)` — any number of leading axes

### The NNX mental model
```python
class SimpleLinear(nnx.Module):
    def __init__(self, din, dout, *, rngs: nnx.Rngs):
        self.w = nnx.Param(jax.random.normal(rngs.params(), (din, dout)))
        ...
    def __call__(self, x):
        return x @ self.w
```

Unlike Flax Linen, NNX modules hold their parameters **directly as attributes**,
so `layer.w` is a live object you can inspect and mutate in place — much closer
to PyTorch's feel, with no separate `params` dict threaded through every call.
The functional purity that `jit` and `grad` need comes from `nnx.split` /
`nnx.merge`, which NNX's own `nnx.jit` and `nnx.grad` apply for you.

`layer.w` is an `nnx.Param`, not a bare array. It forwards `.shape` and the
arithmetic operators, so `x @ self.w` works unchanged, but to read or write the
array itself use **`layer.w[...]`** — `layer.w[...] = new_weights` assigns,
`layer.w[...]` reads. (The old `.value` property still exists but is deprecated
in Flax 0.12.)

Note the shape convention: JAX and Flax use `(din, dout)` and compute `x @ W`,
whereas PyTorch stores `(dout, din)` and computes `x @ W.T`. Mixing them up is
the most common bug when porting weights between the two — and because a
transposed square matrix has the right shape, it fails silently on square layers.

The `1/sqrt(din)` scale is LeCun-style init: it keeps `Var(y) ≈ Var(x)` through
the layer, so activations neither explode nor vanish as you stack them.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class SimpleLinear(nnx.Module):
    """y = x @ w + b"""

    def __init__(self, din: int, dout: int, *, use_bias: bool = True, rngs: nnx.Rngs):
        pass  # Replace this

    def __call__(self, x):
        """(..., din) -> (..., dout)"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax.numpy as jnp
from flax import nnx

layer = SimpleLinear(4, 3, rngs=nnx.Rngs(params=0))
x = jnp.ones((2, 4))

print("w shape:", layer.w.shape, " b shape:", layer.b.shape)
print("out shape:", layer(x).shape)

no_bias = SimpleLinear(4, 3, use_bias=False, rngs=nnx.Rngs(params=0))
print("bias when use_bias=False:", no_bias.b)
print("batched (5, 6, 4) ->", SimpleLinear(4, 3, rngs=nnx.Rngs(params=1))(jnp.ones((5, 6, 4))).shape)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("linear")

# hint("linear")      # stuck? nudge without the answer
# solution("linear")  # spoiler: the reference implementation